# Estandarizando V-Dem

Objetivo general del cuaderno: inspeccionar la codificación de países en `V-Dem-CY-Full+Others-v16.csv` e importar los archivos auxiliares necesarios para su futura estandarización frente al universo de 193 países ONU (`df_regiones_miembros_onu.xlsx`) y frente a los 79 índices intermedios  (`metadatavdem_79_indices_intermedios.xlsx`).


## Bloque 1 — Librerías


In [1]:
import pandas as pd
import os

## Bloque 2 — Carga de columnas identificatorias de V-Dem

**Objetivo:** Definir la ruta del archivo `V-Dem-CY-Full+Others-v16.csv` y cargar únicamente las columnas identificatorias de país (evitando cargar las 4618 columnas del archivo completo).

**Justificación metodológica:** Según la exploración de metadata ya realizada en el notebook 04, las columnas identificatorias de país/tiempo son `country_name`, `country_text_id`, `country_id`, `year` y `COWcode`. Cargar solo estas columnas con `usecols` reduce drásticamente el uso de memoria y es suficiente para el objetivo actual (inspeccionar codificación de países), sin necesidad de traer el resto de indicadores todavía.

**Resultado esperado:** Un DataFrame `df_vdem_paises` con 5 columnas y una fila por país-año (aprox. 202 países × varios años, consistente con lo hallado en notebook 04).

In [2]:
RUTA_VDEM = "V-Dem-CY-Full+Others-v16.csv"
assert os.path.exists(RUTA_VDEM), f"No se encuentra el archivo en: {RUTA_VDEM}"

columnas_identificatorias = ["country_name", "country_text_id", "country_id", "year", "COWcode"]

df_vdem_paises = pd.read_csv(
    RUTA_VDEM,
    usecols=columnas_identificatorias,
    low_memory=False
)

print(df_vdem_paises.shape)
print(df_vdem_paises.dtypes)
df_vdem_paises.head()

(28092, 5)
country_name           str
country_text_id        str
country_id           int64
year                 int64
COWcode            float64
dtype: object


,country_name,country_text_id,country_id,year,COWcode
0,Mexico,MEX,3,1789,70.0
1,Mexico,MEX,3,1790,70.0
2,Mexico,MEX,3,1791,70.0
3,Mexico,MEX,3,1792,70.0
4,Mexico,MEX,3,1793,70.0


## Bloque 3 — Combinaciones únicas de identificación de país

**Objetivo:** Obtener las combinaciones únicas de identificación de país (sin duplicar por año).


**Resultado esperado:** Un DataFrame `df_vdem_paises_unicos` con una fila por combinación única de `country_name`, `country_text_id`, `country_id`, `COWcode`.

**Qué comprobar:** que `country_name`, `country_text_id` y `country_id` den el mismo número de únicos (deberían coincidir 1 a 1 entre sí). Si `country_text_id` tiene menos únicos que `country_name`, hay códigos repetidos para distintos nombres — señal de alerta a investigar antes de cualquier cruce.

**Errores habituales:** `COWcode` puede tener menos únicos que los otros tres (ya vimos en notebook 04 que tiene ~3% de missing y no todas las entidades subnacionales/históricas tienen COW code asignado).

**Cómo verificar:** el conteo de `country_name` únicos debería aproximarse a los 202 confirmados en notebook 04. Si difiere, conviene revisarlo antes de seguir — no asumir que es un error trivial.

In [ ]:
df_vdem_paises_unicos = (
    df_vdem_paises[["country_name", "country_text_id", "country_id", "COWcode"]]
    .drop_duplicates()
    .sort_values("country_name")
    .reset_index(drop=True)
)

print(f"Países únicos (country_name): {df_vdem_paises_unicos['country_name'].nunique()}")
print(f"Códigos únicos (country_text_id): {df_vdem_paises_unicos['country_text_id'].nunique()}")
print(f"IDs únicos (country_id): {df_vdem_paises_unicos['country_id'].nunique()}")

df_vdem_paises_unicos

Países únicos (country_name): 202
Códigos únicos (country_text_id): 202
IDs únicos (country_id): 202


,country_name,country_text_id,country_id,COWcode
0,Afghanistan,AFG,36,700.0
1,Albania,ALB,12,339.0
2,Algeria,DZA,103,615.0
3,Angola,AGO,104,540.0
4,Argentina,ARG,37,160.0
...,...,...,...,...
206,Yemen,YEM,14,679.0
207,Zambia,ZMB,61,551.0
208,Zanzibar,ZZB,236,511.0
209,Zanzibar,ZZB,236,NaN


## Bloque 4 — Importar universo ONU (193 países)

**Objetivo:** Importar `df_regiones_miembros_onu.xlsx`.

**Justificación metodológica:** Es el universo de referencia de 193 países ONU con clasificación M49 e ISO-alpha3, ya construido en el notebook 03. Se carga tal cual, sin transformarlo (principio de mínima intervención), ya que la fusión con V-Dem no está pedida todavía.

**Resultado esperado:** Un DataFrame `df_regiones_onu` con 193 filas y las columnas ya conocidas (`Member State`, `M49_country`, `ISO-alpha3`, `Other Names`, `Country or Area`, `M49_region`, `Region Name`, `M49_subregion`, `Sub-region Name`, `ISO-alpha2 Code`).

**Qué comprobar:** que `df_regiones_onu.shape[0]` sea 193.

**Errores habituales:** ruta relativa incorrecta si el Excel no está en el mismo directorio que el notebook.

In [ ]:
df_regiones_onu = pd.read_excel("df_regiones_miembros_onu.xlsx")

print(df_regiones_onu.shape)
print(df_regiones_onu.dtypes)
df_regiones_onu.head()

(193, 13)
Member State               str
M49_country              int64
ISO-alpha3                 str
Other Names                str
Country or Area            str
M49_region               int64
Region Name                str
M49_subregion            int64
Sub-region Name            str
ISO-alpha2 Code            str
nombre_norm_principal      str
nombre_norm_alt            str
cow_code_countryVdem     int64
dtype: object


,Member State,M49_country,ISO-alpha3,Other Names,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,ISO-alpha2 Code,nombre_norm_principal,nombre_norm_alt,cow_code_countryVdem
0,United States,840,USA,"USA, U.S.A., United States of America",United States of America,19,Americas,21,Northern America,US,united states,usa u s a united states of america,2
1,Australia,36,AUS,Commonwealth of Australia,Australia,9,Oceania,53,Australia and New Zealand,AU,australia,commonwealth of australia,900
2,Djibouti,262,DJI,Republic of Djibouti,Djibouti,2,Africa,202,Sub-Saharan Africa,DJ,djibouti,republic of djibouti,522
3,Ghana,288,GHA,Republic of Ghana,Ghana,2,Africa,202,Sub-Saharan Africa,GH,ghana,republic of ghana,452
4,Kiribati,296,KIR,Republic of Kiribati,Kiribati,9,Oceania,57,Micronesia,KI,kiribati,republic of kiribati,946


###Objetivo: 

Contar cuántos países de V-Dem (identificados por COWcode) encuentran coincidencia en df_regiones_onu (identificados por cow_code_countryVdem), antes de construir Base_VDEM_79.

Justificación metodológica: Es un chequeo de cobertura, no una fusión definitiva todavía. Se trabaja sobre los países únicos de V-Dem. Se convierte COWcode a entero de forma explícita y controlada (con dropna() primero) para poder comparar contra cow_code_countryVdem, que no tiene missing.

Resultado esperado: Un número de coincidencias cercano a 193 (puede ser algo menor si algún país ONU no está en V-Dem, o si hay códigos COW que no calzan exactamente).

In [6]:

# Países únicos de V-Dem con su COWcode (a nivel país, no país-año)
vdem_paises_cow = (
    df_vdem_paises[["country_name", "COWcode"]]
    .drop_duplicates()
    .dropna(subset=["COWcode"])
    .copy()
)
vdem_paises_cow["COWcode"] = vdem_paises_cow["COWcode"].astype(int)

print(f"Países únicos en V-Dem con COWcode no nulo: {vdem_paises_cow['COWcode'].nunique()}")

# Coincidencias: códigos COW de V-Dem que existen en el universo ONU (193 países)
codigos_onu = set(df_regiones_onu["cow_code_countryVdem"])
codigos_vdem = set(vdem_paises_cow["COWcode"])

matches = codigos_vdem & codigos_onu
solo_en_vdem = codigos_vdem - codigos_onu
solo_en_onu = codigos_onu - codigos_vdem

print(f"Coincidencias (match): {len(matches)}")
print(f"Códigos en V-Dem sin match en ONU: {len(solo_en_vdem)}")
print(f"Códigos en ONU sin match en V-Dem: {len(solo_en_onu)}")

Países únicos en V-Dem con COWcode no nulo: 196
Coincidencias (match): 172
Códigos en V-Dem sin match en ONU: 24
Códigos en ONU sin match en V-Dem: 21


## Bloque 5 — Importar metadata de los 79 índices intermedios

**Objetivo:** Importar `metadatavdem_79_indices_intermedios.xlsx`.

**Justificación metodológica:** Contiene la metadata (nombre, pregunta, descripción) de los 79 índices intermedios candidatos. Se carga sin filtrar aún el CSV de V-Dem por estas columnas, tal como se pidió.

**Resultado esperado:** Un DataFrame `df_79_indices` con 79 filas y columnas `columna`, `nombre`, `question`, `description`.

**Qué comprobar:** que `df_79_indices.shape[0]` sea 79.

**Cómo verificar:** revisar que la columna `columna` contenga nombres de variables con el prefijo típico de V-Dem (`v2x...`, `v2xlg...`, etc.), consistentes con lo ya visto en notebook 04.

In [ ]:
df_79_indices = pd.read_excel("metadatavdem_79_indices_intermedios.xlsx")

print(df_79_indices.shape)
df_79_indices.head()

(79, 4)


,columna,nombre,question,description
0,v2x_suffr,Share of population with suffrage,What share of adult citizens as defined by sta...,This question does not take into consideration...
1,v2x_jucon,Judicial constraints on the executive index,To what extent does the executive respect the ...,NaN
2,v2xlg_legcon,Legislative constraints on the executive index,To what extent are the legislature and governm...,The participatory principle of democracy empha...
3,v2x_cspart,Civil society participation index,Are major CSOs routinely consulted by policyma...,The sphere of civil society lies in the public...
4,v2xdd_dd,Direct popular vote index,To what extent is the direct popular vote util...,Direct popular voting refers here to an instit...
